# PySpark + Apache Polaris: e-commerce catalog lab

Use small, hardcoded customers, products and orders to see a catalog in action.
This notebook is self-contained: it does not import the companion script.

**Outcome:** 6 customers, 4 products, 6 orders, total revenue **267.44**.

**Services:** Polaris `localhost:8880`; S3-compatible API `localhost:9001`.
**New demo resources:** `ecomm_demo` catalog and `ecomm-polaris-demo` bucket.

Running the setup cells creates missing demo resources and grants the current
principal access to this catalog. No existing table is dropped.

## 1. Dependencies and a fresh kernel

Install `requirements-lab.txt` before running this notebook. Use Java 17,
PySpark 3.5.6 (Scala 2.12 runtime) and Iceberg 1.9.2. Spark downloads the
matching Iceberg runtime and AWS bundle on first startup.

```bash
python -m pip install -r requirements-lab.txt
```

Start with a fresh kernel. If the driver is inside Docker, replace localhost
with addresses reachable from the driver/executors. Verify that 9001 is the
S3 API, not a storage browser console.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime
from decimal import Decimal
import json
import os
import re
import urllib.error
import urllib.parse
import urllib.request

## 2. Configure the local lab

Defaults include the supplied local demo credentials. Environment variables can
override them. The public configuration printout omits secrets.

The Spark alias `polaris`, the managed Polaris catalog `ecomm_demo`, and the
`s3://...` warehouse location are three separate identifiers.
This lab uses Iceberg **S3FileIO**, not the Hadoop S3A connector.

In [ ]:
@dataclass
class Config:
    base_url: str = 'http://localhost:8880'
    s3_endpoint: str = 'http://localhost:9001'
    client_id: str = 'team'
    client_secret: str = field(default='team1234', repr=False)
    access_key: str = field(default='team', repr=False)
    secret_key: str = field(default='team1234', repr=False)
    catalog: str = 'ecomm_demo'
    bucket: str = 'ecomm-polaris-demo'
    region: str = 'us-east-1'
    realm: str = 'POLARIS'
    master: str = 'local[2]'
    iceberg_version: str = '1.9.2'
    s3_internal_endpoint: str = ''

    def __post_init__(self):
        self.base_url = self.base_url.rstrip('/')
        self.s3_endpoint = self.s3_endpoint.rstrip('/')
        if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', self.catalog):
            raise ValueError('Catalog name must be a simple SQL-safe identifier')
        if not re.fullmatch(r'[a-z0-9][a-z0-9.-]{1,61}[a-z0-9]', self.bucket):
            raise ValueError('Use a valid lowercase S3 bucket name')
        for url in (self.base_url, self.s3_endpoint):
            parsed = urllib.parse.urlparse(url)
            if parsed.scheme not in ('http', 'https') or not parsed.netloc or parsed.username:
                raise ValueError('Endpoints must be HTTP(S) URLs without embedded credentials')

    @property
    def catalog_uri(self):
        return self.base_url + '/api/catalog'

    @property
    def token_uri(self):
        return self.catalog_uri + '/v1/oauth/tokens'

    @property
    def warehouse(self):
        return f's3://{self.bucket}/warehouse/{self.catalog}/'

    @classmethod
    def from_env(cls):
        names = {'base_url':'POLARIS_BASE_URL', 's3_endpoint':'S3_ENDPOINT',
                 'client_id':'POLARIS_CLIENT_ID', 'client_secret':'POLARIS_CLIENT_SECRET',
                 'access_key':'S3_ACCESS_KEY_ID', 'secret_key':'S3_SECRET_ACCESS_KEY',
                 'catalog':'POLARIS_CATALOG_NAME', 'bucket':'S3_BUCKET', 'region':'S3_REGION',
                 'realm':'POLARIS_REALM', 'master':'SPARK_MASTER', 'iceberg_version':'ICEBERG_VERSION',
                 's3_internal_endpoint':'S3_INTERNAL_ENDPOINT'}
        return cls(**{attr:os.environ[name] for attr,name in names.items() if os.environ.get(name)})

    def public_settings(self):
        return {'polaris':self.base_url, 'catalog_api':self.catalog_uri,
                's3_endpoint':self.s3_endpoint, 'polaris_catalog':self.catalog,
                'spark_catalog_alias':'polaris', 'namespace':'ecommerce',
                'warehouse_location':self.warehouse, 'region':self.region,
                'spark_master':self.master, 'iceberg_version':self.iceberg_version}

In [ ]:
config = Config.from_env()
# Or edit endpoints/names explicitly, for example:
# config = Config(base_url="http://localhost:8880", s3_endpoint="http://localhost:9001")
print(json.dumps(config.public_settings(), indent=2))
SKIP_SETUP = False  # True only when bucket, catalog and permissions already exist.

## 3. Authenticate and list Polaris catalogs

Catalog creation/listing uses Polaris's management REST API. Spark SQL does not
create the top-level managed catalog. The client must have management permissions.

`team` is treated as an OAuth client ID; an interactive UI login alone is insufficient.
The token remains private. No automatic retries are made for management mutations.

In [ ]:
class ApiError(RuntimeError):
    def __init__(self, status, method, path):
        self.status = status
        hints = {401:'Check the OAuth client ID/secret and realm.',
                 403:'The client needs management or catalog permissions for this operation.',
                 404:'Check the API route, object name and Polaris version.',
                 409:'Resource conflict: inspect the existing resource before resuming.'}
        super().__init__(f'{method} {path}: HTTP {status}. {hints.get(status, "Check Polaris server logs.")}')

class Polaris:
    def __init__(self, config):
        self.config = config
        self._token = None

    def _request(self, method, url, data=None, form=False, authenticated=True):
        headers = {'Polaris-Realm':self.config.realm}
        if authenticated:
            if not self._token:
                raise RuntimeError('Authenticate before management requests')
            headers['Authorization'] = 'Bearer ' + self._token
        if data is not None:
            headers['Content-Type'] = 'application/x-www-form-urlencoded' if form else 'application/json'
            data = (urllib.parse.urlencode(data) if form else json.dumps(data)).encode()
        request = urllib.request.Request(url, data=data, headers=headers, method=method)
        try:
            with urllib.request.urlopen(request, timeout=10) as response:
                body = response.read()
                return json.loads(body) if body else {}
        except urllib.error.HTTPError as error:
            # Do not echo response bodies: they may contain credentials/configuration.
            raise ApiError(error.code, method, urllib.parse.urlparse(url).path) from None
        except urllib.error.URLError:
            raise RuntimeError(f'Cannot reach {urllib.parse.urlparse(url).netloc}; check service/networking.') from None

    def authenticate(self):
        result = self._request('POST', self.config.token_uri, {
            'grant_type':'client_credentials', 'client_id':self.config.client_id,
            'client_secret':self.config.client_secret, 'scope':'PRINCIPAL_ROLE:ALL'},
            form=True, authenticated=False)
        self._token = result['access_token']
        print('Authenticated to Polaris; token is kept private.')

    def management(self, method, path, payload=None):
        return self._request(method, self.config.base_url + '/api/management/v1' + path, payload)

    def list_catalogs(self):
        catalogs = self.management('GET', '/catalogs').get('catalogs', [])
        print('Polaris catalogs:', [c['name'] for c in catalogs])
        return catalogs

    def ensure_catalog(self):
        """Create the dedicated internal catalog; never overwrite existing storage settings."""
        path = '/catalogs/' + self.config.catalog
        try:
            existing = self.management('GET', path)
        except ApiError as error:
            if error.status != 404:
                raise
            storage = {'storageType':'S3', 'allowedLocations':[self.config.warehouse],
                       'region':self.config.region, 'endpoint':self.config.s3_endpoint,
                       'pathStyleAccess':True, 'stsUnavailable':True}
            if self.config.s3_internal_endpoint:
                storage['endpointInternal'] = self.config.s3_internal_endpoint
            self.management('POST', '/catalogs', {'catalog':{
                'name':self.config.catalog, 'type':'INTERNAL',
                'properties':{'default-base-location':self.config.warehouse},
                'storageConfigInfo':storage}})
            print('Created Polaris catalog:', self.config.catalog)
            return
        if existing.get('type') != 'INTERNAL' or existing.get('properties', {}).get('default-base-location') != self.config.warehouse:
            raise RuntimeError('Existing catalog type/location differs. Choose a new demo catalog or set the matching bucket.')
        if existing.get('storageConfigInfo', {}).get('storageType') != 'S3':
            raise RuntimeError('Existing catalog is not S3-backed')
        print('Reusing existing Polaris catalog:', self.config.catalog)

    def grant_demo_access(self):
        """Assign catalog_admin for this catalog to a dedicated role of the current principal."""
        principals = self.management('GET', '/principals').get('principals', [])
        matched = [p for p in principals if p.get('clientId') == self.config.client_id]
        if len(matched) != 1:
            raise RuntimeError('Cannot uniquely resolve OAuth client to a principal; ask the administrator to grant catalog access, then use --skip-setup.')
        principal = urllib.parse.quote(matched[0]['name'], safe='')
        role = self.config.catalog + '_lab_users'
        try:
            self.management('GET', '/principal-roles/' + role)
        except ApiError as error:
            if error.status != 404:
                raise
            self.management('POST', '/principal-roles', {'principalRole':{'name':role}})
        roles = self.management('GET', f'/principals/{principal}/principal-roles').get('roles', [])
        if role not in [r['name'] for r in roles]:
            self.management('PUT', f'/principals/{principal}/principal-roles', {'principalRole':{'name':role}})
        # Polaris creates catalog_admin with a new internal catalog.
        self.management('GET', f'/catalogs/{self.config.catalog}/catalog-roles/catalog_admin')
        assigned = self.management('GET', f'/principal-roles/{role}/catalog-roles/{self.config.catalog}').get('roles', [])
        if 'catalog_admin' not in [r['name'] for r in assigned]:
            self.management('PUT', f'/principal-roles/{role}/catalog-roles/{self.config.catalog}',
                            {'catalogRole':{'name':'catalog_admin'}})
        print('Demo catalog access ready for the authenticated principal.')

In [ ]:
api = Polaris(config)
api.authenticate()
if not SKIP_SETUP:
    api.list_catalogs()

## 4. Check or create the dedicated S3 bucket

The code checks `ecomm-polaris-demo`; only a missing bucket is created. It does
not remove existing files. Direct local-demo storage credentials are used.
The region defaults to `us-east-1` and can be overridden.

In [ ]:
def s3_client(config):
    import boto3
    from botocore.config import Config as BotoConfig
    return boto3.client('s3', endpoint_url=config.s3_endpoint, region_name=config.region,
                        aws_access_key_id=config.access_key, aws_secret_access_key=config.secret_key,
                        config=BotoConfig(signature_version='s3v4', s3={'addressing_style':'path'},
                                          connect_timeout=5, read_timeout=10, retries={'max_attempts':0}))

def ensure_bucket(config, create=False):
    from botocore.exceptions import ClientError, BotoCoreError
    client = s3_client(config)
    try:
        client.head_bucket(Bucket=config.bucket)
        print('S3 bucket reachable:', config.bucket)
        return True
    except ClientError as error:
        status = error.response.get('ResponseMetadata', {}).get('HTTPStatusCode')
        if status == 404:
            if not create:
                print('Demo bucket does not exist yet:', config.bucket)
                return False
            payload = {'Bucket':config.bucket}
            if config.region != 'us-east-1':
                payload['CreateBucketConfiguration'] = {'LocationConstraint':config.region}
            client.create_bucket(**payload)
            print('Created dedicated demo bucket:', config.bucket)
            return True
        raise RuntimeError(f'S3 bucket check returned HTTP {status}; verify API port and credentials.') from None
    except BotoCoreError:
        raise RuntimeError('S3 API check failed. Verify that port 9001 is the S3 API, not a console, and is reachable.') from None

In [ ]:
bucket_ready = ensure_bucket(config, create=not SKIP_SETUP)
if not bucket_ready:
    raise RuntimeError("The existing bucket is required when SKIP_SETUP=True")

## 5. Create a managed catalog and give the current identity access

The management payload links an internal S3 catalog to its default warehouse
location. An existing catalog is reused only when its type and location match.

The dedicated principal role `ecomm_demo_lab_users` receives this catalog's
`catalog_admin` role and is attached to the principal matching the OAuth client.
No other principal's credentials are changed.

Credential vending is disabled for new demo catalogs (`stsUnavailable=True`).
Polaris may still need server-side storage credentials and an internal endpoint
configured by your service deployment.

In [ ]:
if not SKIP_SETUP:
    api.ensure_catalog()
    api.grant_demo_access()
    catalogs = api.list_catalogs()
else:
    print("Using the pre-provisioned catalog and grants")

## 6. Start Spark with an Iceberg REST catalog

Notice `.type=rest`, `.warehouse=ecomm_demo`, the explicit token endpoint and
the S3FileIO settings. The Iceberg runtime and AWS bundle versions match.
No `hadoop-aws` JAR or `fs.s3a.*` settings are used.

Do not print Spark's full configuration: it contains local authentication secrets.

In [ ]:
def start_spark(config, alias='polaris'):
    import pyspark
    from pyspark.sql import SparkSession
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', alias):
        raise ValueError('Spark catalog alias must be a simple identifier')
    if not pyspark.__version__.startswith('3.5.'):
        raise RuntimeError('This lab uses Spark 3.5.x / Scala 2.12. Install requirements-lab.txt or adapt the runtime coordinates.')
    if SparkSession.getActiveSession() is not None:
        raise RuntimeError('A Spark session is already active. Restart the notebook kernel so Iceberg JARs/extensions load before Spark starts.')
    packages = ','.join([
        f'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{config.iceberg_version}',
        f'org.apache.iceberg:iceberg-aws-bundle:{config.iceberg_version}'])
    settings = {
        'spark.jars.packages':packages,
        'spark.sql.extensions':'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions',
        'spark.sql.catalog.polaris':'org.apache.iceberg.spark.SparkCatalog',
        'spark.sql.catalog.polaris.type':'rest',
        'spark.sql.catalog.polaris.uri':config.catalog_uri,
        'spark.sql.catalog.polaris.warehouse':config.catalog,
        'spark.sql.catalog.polaris.oauth2-server-uri':config.token_uri,
        'spark.sql.catalog.polaris.credential':config.client_id + ':' + config.client_secret,
        'spark.sql.catalog.polaris.scope':'PRINCIPAL_ROLE:ALL',
        'spark.sql.catalog.polaris.header.Polaris-Realm':config.realm,
        'spark.sql.catalog.polaris.io-impl':'org.apache.iceberg.aws.s3.S3FileIO',
        'spark.sql.catalog.polaris.s3.endpoint':config.s3_endpoint,
        'spark.sql.catalog.polaris.s3.path-style-access':'true',
        'spark.sql.catalog.polaris.s3.access-key-id':config.access_key,
        'spark.sql.catalog.polaris.s3.secret-access-key':config.secret_key,
        'spark.sql.catalog.polaris.client.region':config.region,
        'spark.sql.shuffle.partitions':'2', 'spark.sql.session.timeZone':'UTC',
        'spark.redaction.regex':'(?i)secret|password|token|access[.]?key|credential'}
    # Direct local-demo storage credentials; do not request vended credentials.
    settings = {name.replace('spark.sql.catalog.polaris', 'spark.sql.catalog.' + alias):value
                for name,value in settings.items()}
    builder = SparkSession.builder.master(config.master).appName('Polaris E-commerce Catalog Lab')
    for name, value in settings.items():
        builder = builder.config(name, value)
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark

def show_sql(spark, query):
    print('\nSQL>', query.strip())
    result = spark.sql(query)
    result.show(30, truncate=False)
    return result

In [ ]:
spark = start_spark(config)

## 7. Create namespaces and tables; list and describe them

Read the SQL in the next cell: `SHOW CATALOGS`, `CREATE NAMESPACE`,
`SHOW NAMESPACES`, `CREATE TABLE ... USING iceberg`, `SHOW TABLES`, and
`DESCRIBE TABLE [EXTENDED]`.

`SHOW CATALOGS` lists Spark catalog aliases, while the earlier management call
lists Polaris catalogs. An ownership property prevents accidental seeding of
an unrelated existing table.

In [ ]:
def create_tables(spark):
    show_sql(spark, 'SHOW CATALOGS')
    show_sql(spark, 'CREATE NAMESPACE IF NOT EXISTS polaris.ecommerce')
    show_sql(spark, 'SHOW NAMESPACES IN polaris')
    tables = {
        'customers':'customer_id BIGINT, customer_name STRING, email STRING, country STRING, created_at TIMESTAMP',
        'products':'product_id BIGINT, product_name STRING, category STRING, unit_price DECIMAL(10,2)',
        'orders':'order_id BIGINT, customer_id BIGINT, product_id BIGINT, quantity INT, order_date DATE'}
    for table, schema in tables.items():
        show_sql(spark, f"CREATE TABLE IF NOT EXISTS polaris.ecommerce.{table} ({schema}) USING iceberg TBLPROPERTIES ('format-version'='2', 'demo.owner'='polaris_ecommerce_lab')")
        properties = dict(spark.sql(f'SHOW TBLPROPERTIES polaris.ecommerce.{table}').collect())
        if properties.get('demo.owner') != 'polaris_ecommerce_lab':
            raise RuntimeError(f'{table} is not marked as this lab table; choose another demo catalog.')
    show_sql(spark, 'SHOW TABLES IN polaris.ecommerce')
    show_sql(spark, 'DESCRIBE TABLE polaris.ecommerce.customers')
    show_sql(spark, 'DESCRIBE TABLE EXTENDED polaris.ecommerce.customers')

In [ ]:
create_tables(spark)

## 8. Hardcoded DataFrames → persistent Iceberg tables

Five initial customers, four products and six orders are declared below in
Python. Prices use `Decimal`; schemas are explicit. The DataFrames become
temporary source views for keyed `MERGE` statements.

The source views exist only in Spark; the target Iceberg tables persist in S3
and are registered in Polaris. Repeated seeding updates matching demo IDs
and inserts missing ones without duplicating them.

In [ ]:
def seed_data(spark):
    from datetime import date
    customers = [
        (1,'Asha Rao','asha@example.test','India',datetime(2026,9,1,9)),
        (2,'Ben Carter','ben@example.test','UK',datetime(2026,9,2,9)),
        (3,'Carla Silva','carla@example.test','Brazil',datetime(2026,9,3,9)),
        (4,'Dev Shah','dev@example.test','India',datetime(2026,9,4,9)),
        (5,'Emma Brown','emma@example.test','USA',datetime(2026,9,5,9))]
    products = [(1,'Travel Mug','Kitchen',Decimal('19.99')),
                (2,'Laptop Backpack','Bags',Decimal('49.50')),
                (3,'Notebook','Stationery',Decimal('9.99')),
                (4,'Desk Lamp','Lighting',Decimal('14.50'))]
    orders = [(101,1,1,2,date(2026,9,10)), (102,2,2,1,date(2026,9,10)),
              (103,1,3,3,date(2026,9,11)), (104,3,1,1,date(2026,9,11)),
              (105,4,4,2,date(2026,9,12)), (106,5,2,2,date(2026,9,12))]
    definitions = [
        ('customers','customer_id',customers,'customer_id long, customer_name string, email string, country string, created_at timestamp'),
        ('products','product_id',products,'product_id long, product_name string, category string, unit_price decimal(10,2)'),
        ('orders','order_id',orders,'order_id long, customer_id long, product_id long, quantity int, order_date date')]
    for table, key, rows, schema in definitions:
        df = spark.createDataFrame(rows, schema)
        print('\nHardcoded source:', table)
        df.show(truncate=False)
        df.createOrReplaceTempView('seed_' + table)
        columns = df.columns
        updates = ', '.join(f't.{c} = s.{c}' for c in columns if c != key)
        # Explicit columns allow later schema evolution without wildcard mismatch.
        show_sql(spark, f'''MERGE INTO polaris.ecommerce.{table} t USING seed_{table} s
            ON t.{key} = s.{key}
            WHEN MATCHED THEN UPDATE SET {updates}
            WHEN NOT MATCHED THEN INSERT ({', '.join(columns)})
            VALUES ({', '.join('s.'+c for c in columns)})''')

In [ ]:
seed_data(spark)

## 9. Evolve the schema and demonstrate MERGE

`ALTER TABLE ... ADD COLUMN` adds `loyalty_tier` once. A second MERGE updates
Asha (ID 1) to gold and inserts Farah (ID 6) as silver. Rerunning still yields
six customers. Existing customers without a tier retain null.

Spark/Iceberg perform the data operation; Polaris coordinates the catalog commit.

In [ ]:
def evolve_and_merge(spark):
    if 'loyalty_tier' not in spark.table('polaris.ecommerce.customers').columns:
        show_sql(spark, 'ALTER TABLE polaris.ecommerce.customers ADD COLUMN loyalty_tier STRING')
    changes = [(1,'Asha Rao','asha@example.test','India',datetime(2026,9,1,9),'gold'),
               (6,'Farah Khan','farah@example.test','India',datetime(2026,9,13,9),'silver')]
    spark.createDataFrame(changes, 'customer_id long, customer_name string, email string, country string, created_at timestamp, loyalty_tier string').createOrReplaceTempView('customer_changes')
    show_sql(spark, '''MERGE INTO polaris.ecommerce.customers t USING customer_changes s
        ON t.customer_id = s.customer_id
        WHEN MATCHED THEN UPDATE SET t.loyalty_tier = s.loyalty_tier
        WHEN NOT MATCHED THEN INSERT *''')
    show_sql(spark, 'DESCRIBE TABLE polaris.ecommerce.customers')
    show_sql(spark, 'SHOW TBLPROPERTIES polaris.ecommerce.customers')

In [ ]:
evolve_and_merge(spark)

## 10. Query, describe metadata and verify the result

Join orders to customers and products, aggregate country revenue, inspect
Iceberg's `.snapshots` and `.files` metadata tables, and check the result.

Expected: **6 / 4 / 6** rows, unique identifiers, no missing references, and
**267.44** total revenue. Iceberg does not enforce relational foreign keys;
the checks below verify those relationships for our sample.

In [ ]:
def inspect_and_verify(spark):
    show_sql(spark, 'SELECT * FROM polaris.ecommerce.customers ORDER BY customer_id')
    show_sql(spark, '''SELECT c.customer_name, p.product_name, o.quantity,
        p.unit_price, CAST(o.quantity * p.unit_price AS DECIMAL(12,2)) AS order_amount
        FROM polaris.ecommerce.orders o
        JOIN polaris.ecommerce.customers c ON c.customer_id = o.customer_id
        JOIN polaris.ecommerce.products p ON p.product_id = o.product_id
        ORDER BY o.order_id''')
    show_sql(spark, '''SELECT c.country, COUNT(*) AS order_count,
        SUM(o.quantity * p.unit_price) AS revenue
        FROM polaris.ecommerce.orders o
        JOIN polaris.ecommerce.customers c ON c.customer_id = o.customer_id
        JOIN polaris.ecommerce.products p ON p.product_id = o.product_id
        GROUP BY c.country ORDER BY revenue DESC''')
    show_sql(spark, 'SELECT committed_at, snapshot_id, operation FROM polaris.ecommerce.customers.snapshots ORDER BY committed_at')
    show_sql(spark, 'SELECT file_path, record_count FROM polaris.ecommerce.customers.files')
    failures = []
    for table, count, key in [('customers',6,'customer_id'),('products',4,'product_id'),('orders',6,'order_id')]:
        row = spark.sql(f'SELECT COUNT(*) AS n, COUNT(DISTINCT {key}) AS unique_n FROM polaris.ecommerce.{table}').first()
        if row.n != count or row.unique_n != count:
            failures.append(f'{table}: expected {count} unique rows, got {row.n}/{row.unique_n}')
    revenue = spark.sql('''SELECT SUM(o.quantity * p.unit_price) AS revenue
        FROM polaris.ecommerce.orders o JOIN polaris.ecommerce.products p
        ON p.product_id = o.product_id''').first().revenue
    if revenue != Decimal('267.44'):
        failures.append(f'Revenue differs from expected 267.44: {revenue}')
    orphans = spark.sql('''SELECT o.order_id FROM polaris.ecommerce.orders o
        LEFT JOIN polaris.ecommerce.customers c ON o.customer_id=c.customer_id
        LEFT JOIN polaris.ecommerce.products p ON o.product_id=p.product_id
        WHERE c.customer_id IS NULL OR p.product_id IS NULL''').count()
    if orphans:
        failures.append('Orders have missing customer/product references')
    if failures:
        raise AssertionError('; '.join(failures))
    print('\nPASS: 6 customers, 4 products, 6 orders; unique IDs; no orphan orders; revenue 267.44.')

In [ ]:
inspect_and_verify(spark)

## 11. Additional catalog discovery exercises

These cells are read-only. Compare the namespace/table lists with the printed
management catalog list. `USE` selects Spark's configured catalog namespace.

In [ ]:
show_sql(spark, "USE polaris.ecommerce")
show_sql(spark, "SELECT current_catalog(), current_database()")
show_sql(spark, "SHOW TABLES")
show_sql(spark, "DESCRIBE NAMESPACE EXTENDED polaris.ecommerce")
show_sql(spark, "SELECT * FROM polaris.ecommerce.orders ORDER BY order_id")

## 12. Stop compute; retain the tables

Stopping Spark does not delete the bucket, catalog or tables. A later session
configured with the same Polaris catalog can discover them.

No DROP statement is included in Run All. This notebook intentionally retains
the resources for subsequent lessons.

In [ ]:
spark.stop()

## Reruns and limits

Use this dedicated demo catalog only. Seeding restores base values for matching
demo IDs; extra unrelated rows are retained and cause validation to fail.
Polaris/S3 must be reachable and Java/connector dependencies must be present.

This notebook has no fabricated execution outputs. The authoring session could
not reach the endpoints and had no installed PySpark. Offline structure and
mocked setup checks do not replace an end-to-end run on your stack.

## Official references

- [Polaris management API](https://github.com/apache/polaris/blob/main/spec/polaris-management-service.yml)
- [S3-compatible catalog setup](https://polaris.apache.org/releases/1.8.0/getting-started/creating-a-catalog/s3/catalog-minio/)
- [Spark catalog configuration](https://iceberg.apache.org/docs/latest/spark-configuration/)
- [Spark DDL](https://iceberg.apache.org/docs/latest/spark-ddl/)
- [Spark MERGE / writes](https://iceberg.apache.org/docs/latest/spark-writes/)
- [S3FileIO](https://iceberg.apache.org/docs/latest/aws/)

See `02_polaris_ecommerce_lab.md` for commands, environment overrides and troubleshooting.